# Inverse Kinematics of a 5-DOF Robotic Arm

This notebook calculates the joint angles ($q_1$ to $q_5$) required to place a robotic end-effector at a specific Cartesian coordinate ($X, Y, Z$) with a specific pitch and roll.

### Step 1: Initialization and Target Coordinates
First, we define our target point and the physical lengths and offsets of the robot's links. We use a logical sequence from base to gripper ($l_2, l_3, l_4$).

In [2]:
import math

# Target Point Inputs
X = 0.15
Y = 0.15
Z = 0.35
theta_pitch = 0.0
roll = 0.0

# Link Lengths
l2 = 0.1160 # Upper arm length
l3 = 0.1350 # Lower arm (forearm) length
l4 = 0.1351 # Distance from wrist to gripper center

### Step 2: Base Angle ($q_1$) and Cartesian to Cylindrical Mapping


To simplify the 3D problem into a 2D planar problem, we determine the base rotation ($q_1$) and map our coordinates into a cylindrical $R-Z$ plane aligned with the arm. 

We account for the physical offsets of the shoulder joint ($y_1, y_2, z_1, z_2$) to find the target radius ($r_{target}$) and height ($z_{target}$) relative to the **upper arm's origin**.

$$r_{ee} = \sqrt{X^2 + (Y - y_1)^2}$$

In [3]:
# Y-direction offsets
y1 = 0.0452 # Offset in the y-direction of the shoulder
y2 = 0.0306 # Offset from the shoulder to the upper arm

# Base angle calculation
q1 = math.atan2(Y - y1, X) - (math.pi / 2)

# End effector radius around the shoulder
r_ee = math.sqrt(X**2 + (Y - y1)**2)
r_target = r_ee - y2 # Radius relative to the upper arm

# Z-direction offsets
z1 = 0.0165 # Base to shoulder
z2 = 0.1025 # Shoulder to upper arm

z_target = Z - (z1 + z2) # Height relative to the upper arm

### Step 3: Kinematic Decoupling (Finding the Wrist Center)
Because the last joint (roll) doesn't affect the position, and the pitch angle only affects the orientation of the final link ($l_4$), we can "decouple" the position and orientation. We project backward from the end-effector to find the position of the **wrist center** ($r_w, z_w$).

$$r_w = r_{target} - l_4 \cos(\theta_{pitch})$$
$$z_w = z_{target} - l_4 \sin(\theta_{pitch})$$

In [4]:
# Calculate Wrist Center coordinates in the R-Z plane
r_w = r_target - l4 * math.cos(theta_pitch)
z_w = z_target - l4 * math.sin(theta_pitch)

### Step 4: Planar 2R Inverse Kinematics


Now the problem is reduced to a simple 2-link planar arm reaching for the wrist center ($r_w, z_w$). We use the **Law of Cosines** to find the elbow angle ($\theta_3$).

$$\cos(\theta_3) = \frac{r_w^2 + z_w^2 - l_2^2 - l_3^2}{2 l_2 l_3}$$

*Note: We clamp $\cos(\theta_3)$ between $-1$ and $1$ to prevent math domain errors if floating-point limits push the target infinitesimally out of reach.*

In [5]:
# Apply Law of Cosines
cos3 = (r_w**2 + z_w**2 - l2**2 - l3**2) / (2 * l2 * l3)

# Safety Clamp to prevent imaginary numbers
cos3 = max(min(cos3, 1.0), -1.0)

# Calculate sine using the Pythagorean identity
sin3 = math.sqrt(1 - cos3**2)

# Calculate the planar elbow angle
theta_3 = math.atan2(sin3, cos3)

### Step 5: Calculating the Shoulder Angle ($\theta_2$)
With the elbow angle known, we can find the angle of the upper arm ($\theta_2$) relative to the horizon. We project the links onto the axes to find the components $k_1$ and $k_2$.

$$k_1 = l_2 + l_3 \cos(\theta_3)$$
$$k_2 = l_3 \sin(\theta_3)$$
$$\theta_2 = \text{atan2}(z_w, r_w) + \text{atan2}(k_2, k_1)$$

In [6]:
# Forward kinematics components of the 2-link chain
k1 = l2 + l3 * cos3
k2 = l3 * sin3

# Calculate the planar shoulder angle
theta_2 = math.atan2(z_w, r_w) + math.atan2(k2, k1)

### Step 6: Joint Offsets and Final Mapping
Physical robots rarely have their zero-angle aligned perfectly with the theoretical math horizon. We apply the specific hardware offsets to map our theoretical angles ($\theta_2, \theta_3$) to the actual motor joints ($q_2, q_3$). Finally, we calculate $q_4$ to achieve the desired pitch.

In [7]:
# Hardware offset for the upper-to-lower arm link
theta_2_off = math.atan2(0.11257, 0.028)
theta_3_off = math.atan2(0.0052, 0.1349)


# Map theoretical angles to actual motor joints
q2 = theta_2 - theta_2_off
q3 = theta_2_off - theta_3 - theta_3_off
q4 = theta_pitch - q2 - q3
q5 = roll

# Compile the final joint array
q = [q1, q2, q3, q4, q5]

# Print neatly formatted results
print("Computed Joint Angles (radians):")
for i, angle in enumerate(q):
    print(f"q{i+1}: {angle:.4f}")

Computed Joint Angles (radians):
q1: -0.9610
q2: 0.5973
q3: 0.4955
q4: -1.0928
q5: 0.0000
